# Solutions — SQL and DataFrame API (Hard 12)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_franchises   = spark.table("samples.bakehouse.sales_franchises")
df_customers    = spark.table("samples.bakehouse.sales_customers")
df_orders       = spark.table("samples.tpch.orders")
df_customer     = spark.table("samples.tpch.customer")
df_bookings     = spark.table("samples.wanderbricks.bookings")

# Register temp views so all tables are queryable via spark.sql
df_transactions.createOrReplaceTempView("transactions")
df_franchises.createOrReplaceTempView("franchises")
df_customers.createOrReplaceTempView("customers")
df_orders.createOrReplaceTempView("orders")
df_customer.createOrReplaceTempView("tpch_customer")
df_bookings.createOrReplaceTempView("bookings")

## Solution 1 — Register Views and Query with SQL

In [ ]:
result_1 = spark.sql("""
    SELECT   f.franchiseID,
             f.name         AS franchiseName,
             SUM(t.totalPrice)     AS total_revenue,
             COUNT(*)              AS transaction_count
    FROM     transactions t
    JOIN     franchises    f  ON t.franchiseID = f.franchiseID
    GROUP BY f.franchiseID, f.name
    ORDER BY total_revenue DESC
""")

## Solution 2 — SQL CTEs for Multi-Step Logic

In [ ]:
result_2 = spark.sql("""
    WITH daily AS (
        SELECT CAST(dateTime AS DATE) AS sale_date,
               product,
               SUM(totalPrice) AS daily_revenue
        FROM   transactions
        GROUP BY 1, 2
    ),
    ranked AS (
        SELECT sale_date, product, daily_revenue,
               RANK() OVER (PARTITION BY sale_date ORDER BY daily_revenue DESC) AS rnk
        FROM daily
    )
    SELECT sale_date, product, daily_revenue
    FROM   ranked
    WHERE  rnk = 1
    ORDER BY sale_date DESC
""")

## Solution 3 — SQL Window Functions

In [ ]:
result_3 = spark.sql("""
    WITH ranked AS (
        SELECT franchiseID,
               transactionID,
               product,
               totalPrice,
               RANK() OVER (PARTITION BY franchiseID ORDER BY totalPrice DESC) AS rnk
        FROM   transactions
    )
    SELECT franchiseID, transactionID, product, totalPrice, rnk
    FROM   ranked
    WHERE  rnk <= 3
""")

## Solution 4 — DataFrame API vs SQL: Prove They're Equivalent

In [ ]:
# DataFrame API version
total = df_transactions.count()

result_4 = (
    df_transactions
    .groupBy("paymentMethod")
    .agg(F.count("*").alias("transaction_count"))
    .withColumn("pct_share",
        F.round(F.col("transaction_count") / F.lit(total) * 100, 2)
    )
    .orderBy("paymentMethod")
)

# SQL version (equivalent)
result_4_sql = spark.sql("""
    SELECT paymentMethod,
           COUNT(*) AS transaction_count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_share
    FROM   transactions
    GROUP BY paymentMethod
    ORDER BY paymentMethod
""")

## Solution 5 — SQL Subquery Pattern (IN / EXISTS)

In [ ]:
result_5 = spark.sql("""
    SELECT o_orderkey, o_custkey, o_totalprice, o_orderdate
    FROM   orders
    WHERE  o_custkey IN (
        SELECT c_custkey FROM tpch_customer WHERE c_mktsegment = 'BUILDING'
    )
""")

## Solution 6 — Explore the Catalog with spark.catalog

In [ ]:
result_6 = spark.sql("SHOW TABLES IN samples.bakehouse")

## Solution 7 — DESCRIBE TABLE for Schema Metadata

In [ ]:
result_7 = spark.sql("DESCRIBE TABLE samples.bakehouse.sales_transactions")

## Solution 8 — Dynamic SQL Generation

In [ ]:
def group_and_count(table_name, group_cols):
    cols_str = ", ".join(group_cols)
    sql = f"SELECT {cols_str}, COUNT(*) AS row_count FROM {table_name} GROUP BY {cols_str} ORDER BY row_count DESC"
    return spark.sql(sql)

result_8 = group_and_count('transactions', ['product', 'paymentMethod'])